# Rede Neural de Tradução LIBRAS para Português (CNN + RNN + GPT)

Este notebook implementa um pipeline de tradução de LIBRAS para português utilizando PyTorch, com entrada de sequências de fingerlandmarks extraídas por MediaPipe. O pipeline inclui:
- CNN para extração espacial dos landmarks
- RNN (LSTM/GRU) para modelagem temporal e tradução
- Polimento da frase com API GPT/Gemini (opcional)

## 1. Instalação das Bibliotecas Necessárias

Instale as bibliotecas essenciais para o projeto. Adapte conforme necessário para seu ambiente.

In [42]:
# # Instalação das bibliotecas (Google Colab)
# %pip install opencv-python mediapipe pandas numpy pyarrow streamlit Pillow torch torchvision torchaudio transformers tqdm scikit-learn openai

## 2. Configuração de Dispositivo

Verifique se CUDA está disponível para acelerar o treinamento.

In [43]:
import torch

# print("CUDA disponível:", torch.cuda.is_available())
# if torch.cuda.is_available():
#     print("Nome da GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cpu


## 3. Transformação e Carregamento dos Dados

Crie um Dataset e DataLoader PyTorch para ler o arquivo `labels_metadata.csv` com referências para os arquivos de landmarks, e apos isso faz uma tokenização.

In [44]:
# 3. Transformação e Carregamento dos Dados
import os
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class FingerLandmarksDataset(Dataset):
    def __init__(self, csv_path, landmarks_dir):
        self.data = pd.read_csv(csv_path)
        self.landmarks_dir = landmarks_dir

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        # Compatível com o padrão do create_landmarks.ipynb
        landmark_path = row['landmark_path'] if 'landmark_path' in row else row['path']
        # Se o caminho já for absoluto ou relativo, não concatena
        if not os.path.isabs(landmark_path) and not landmark_path.startswith('.'):
            landmark_path = os.path.join(self.landmarks_dir, landmark_path)
        arr = np.load(landmark_path, allow_pickle=True)
        # Se for dict (mãos + braços), pega apenas as mãos
        if isinstance(arr, np.ndarray) and arr.dtype == object:
            arr = arr.item()
        if isinstance(arr, dict) and 'hands' in arr:
            hands = arr['hands']
            # Se for sequência temporal, shape (frames, 2, 21, 3) → (frames, 21, 3) para cada mão
            if hands.ndim == 4:
                hands = hands.reshape(-1, 21, 3)
            # Se for imagem, shape (2, 21, 3) → (1, 2, 21, 3)
            elif hands.ndim == 3:
                hands = hands[np.newaxis, ...]
            landmarks = hands
        else:
            # Compatibilidade com arquivos antigos: (seq_len, 21, 3) ou (2, 21, 3)
            if arr.ndim == 3 and arr.shape[0] == 2 and arr.shape[1] == 21:
                # (2, 21, 3) → (1, 2, 21, 3)
                arr = arr[np.newaxis, ...]
            landmarks = arr
        # Garante shape (seq_len, 21, 3)
        if landmarks.ndim == 4:
            landmarks = landmarks.reshape(-1, 21, 3)
        phrase = row['phrase']
        return torch.tensor(landmarks, dtype=torch.float32), phrase

# Instancia o dataset
csv_path = '../dataset/processed/labels_metadata.csv'
landmarks_dir = '../dataset/processed/landmarks/'
dataset = FingerLandmarksDataset(csv_path, landmarks_dir)

In [45]:
# 3.1 Tokenização: Customizada vs Pré-treinada (HuggingFace)
# Tokenização recomendada: HuggingFace BERTimbau (WordPiece para português)
from transformers import BertTokenizer

hf_tokenizer = BertTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')

def tokenize(text):
    return hf_tokenizer.tokenize(text)

def encode(text, vocab=None):
    ids = hf_tokenizer.encode(text, add_special_tokens=False)
    return ids

# # Tokenização customizada (split por espaço)
# from collections import Counter
# def tokenize(text):
#     return text.lower().split()
# def encode(text, vocab):
#     return [vocab.get(token, vocab['<unk>']) for token in tokenize(text)]

print(f"Exemplo de tokenização HuggingFace: {tokenize('Eu estou bem!')}")
print(f"Exemplo de encoding HuggingFace: {encode('Eu estou bem!')}")

Exemplo de tokenização HuggingFace: ['Eu', 'estou', 'bem', '!']
Exemplo de encoding HuggingFace: [3396, 12044, 1004, 106]


In [46]:
# 3.2 DataLoader com collate_fn para tokenização dinâmica

def collate_fn(batch):
    inputs, phrases = zip(*batch)
    inputs = torch.stack(inputs)
    encoded = [torch.tensor(encode(phrase)) for phrase in phrases]
    targets = pad_sequence(encoded, batch_first=True, padding_value=0)  # 0 é o <pad> do BERT
    return inputs, targets

batch_size = 32  # Certifique-se de definir batch_size antes
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

## 4. Arquitetura da Rede Neural

A arquitetura consiste em:
- Um módulo CNN para extrair representações espaciais dos landmarks de cada frame.
- Um módulo RNN (LSTM) para modelar a sequência temporal e gerar a frase.

In [47]:
# 4. Arquitetura da Rede Neural
import torch.nn as nn

class CNNEncoder(nn.Module):
    def __init__(self, n_points, emb_size):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=3, out_channels=32, kernel_size=1)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(32, 64, 1)
        self.fc = nn.Linear(64 * n_points, emb_size)

    def forward(self, x):
        # Garante shape (batch, seq_len, 21, 3)
        if x.ndim == 3:
            x = x.unsqueeze(1)
        batch, seq_len, n_points, _ = x.shape
        x = x.view(-1, n_points, 3).permute(0, 2, 1)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        x = x.view(batch, seq_len, -1)
        return x

class Sign2TextModel(nn.Module):
    def __init__(self, n_points, emb_size, hidden_size, vocab_size, num_layers=1, dropout=0.5):
        super().__init__()
        self.encoder = CNNEncoder(n_points, emb_size)
        self.rnn = nn.LSTM(emb_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.encoder(x)
        out, _ = self.rnn(x)
        out = self.fc(out)
        return out

#### Descrição da Arquitetura:

A rede neural implementada neste projeto segue um modelo híbrido CNN + RNN para tradução de LIBRAS para português. Abaixo está um detalhamento da arquitetura:

- **Entrada:**  
  Sequência de landmarks da mão  
  Shape: `(batch_size, seq_len, 21, 3)`  
  (onde 21 = número de pontos da mão, 3 = coordenadas x, y, z)

- **Camada Oculta 1 (CNNEncoder):**  
  - Conv1d: 3 canais → 32 filtros, kernel=1, ativação ReLU  
  - Conv1d: 32 filtros → 64 filtros, kernel=1, ativação ReLU  
  - Linear: 64 * 21 → 128 (emb_size), ativação implícita  
  (Extrai características espaciais de cada frame)

- **Camada Oculta 2 (LSTM):**  
  - LSTM: entrada 128 (emb_size), saída 256 (hidden_size), 1 camada, dropout 0.5  
  (Modela dependências temporais entre frames)

- **Saída:**  
  - Linear: 256 (hidden_size) → vocab_size (tamanho do vocabulário)  
  (Para cada passo da sequência, prevê um token da frase em português)

---

O que cada função faz:

- **ReLU:** Função de ativação que mantém valores positivos e zera os negativos, acelerando o aprendizado e evitando problemas de gradiente.
- **LSTM:** Tipo de RNN que aprende dependências de longo prazo em sequências, essencial para entender o contexto temporal dos sinais.
- **Linear:** Camada totalmente conectada que transforma o vetor oculto em logits para cada token do vocabulário.
- **Dropout:** Técnica de regularização que desativa aleatoriamente neurônios durante o treino, ajudando a evitar overfitting.

---

**Resumo do fluxo:**  
Landmarks → CNN (features espaciais) → LSTM (contexto temporal) → Linear (previsão de tokens da frase).

## 5. Configuração de Hiperparâmetros

Defina os hiperparâmetros principais para o treinamento.

In [48]:
# 5. Configuração de Hiperparâmetros
num_epochs = 50
batch_size = 32
learning_rate = 0.001
dropout = 0.5
emb_size = 128
hidden_size = 256
num_layers = 1
n_points = 21  # Número de pontos da mão
vocab_size = hf_tokenizer.vocab_size  # Usa vocab_size do BERT

## 6. Inicialização dos Pesos

Utilize Xavier para tanh e He para ReLU.

In [49]:
# 6. Inicialização dos Pesos

def init_weights(m):
    if isinstance(m, nn.Conv1d) or isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.LSTM):
        for name, param in m.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                nn.init.zeros_(param.data)

model = Sign2TextModel(n_points, emb_size, hidden_size, vocab_size, num_layers, dropout)
model.apply(init_weights)
model = model.to(device)

c:\Files\zPessoal\ifb\ifb_tcc\venv\lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn(


## 7. Função de Perda e Otimizador

Utilize CrossEntropyLoss para classificação de palavras/frases e Adam como otimizador.

In [50]:
# 7. Função de Perda e Otimizador
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

## 8. Loop de Treinamento

Loop clássico de treinamento: forward, backward, otimização, checkpoint e early stopping.

In [55]:
# 8. Loop de Treinamento
import copy

def train_model(model, train_loader, criterion, optimizer, device, num_epochs):
    best_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            inputs, targets = batch
            inputs = inputs.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)  # outputs: (batch, seq_len, vocab_size)
            batch_size, seq_len, vocab_size = outputs.shape
            outputs_flat = outputs.reshape(-1, vocab_size)      # (batch*seq_len, vocab_size)
            targets_flat = targets.reshape(-1)                  # (batch*seq_len)
            mask = targets_flat != 0                            # (batch*seq_len)
            if mask.sum() == 0:
                continue  # ignora batches só de padding
            # Seleciona apenas os índices válidos
            outputs_valid = outputs_flat[mask.nonzero(as_tuple=True)[0]]
            targets_valid = targets_flat[mask]
            loss = criterion(outputs_valid, targets_valid.long())
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), '../models/best_model.pt')
    print("Treinamento finalizado.")
    model.load_state_dict(best_model_wts)
    return model

model = train_model(model, train_loader, criterion, optimizer, device, num_epochs)

Epoch 1/50 - Loss: 3.3187
Epoch 2/50 - Loss: 2.5511
Epoch 2/50 - Loss: 2.5511
Epoch 3/50 - Loss: 2.5517
Epoch 3/50 - Loss: 2.5517
Epoch 4/50 - Loss: 2.5538
Epoch 4/50 - Loss: 2.5538
Epoch 5/50 - Loss: 2.5524
Epoch 5/50 - Loss: 2.5524
Epoch 6/50 - Loss: 2.5569
Epoch 6/50 - Loss: 2.5569
Epoch 7/50 - Loss: 2.5498
Epoch 7/50 - Loss: 2.5498
Epoch 8/50 - Loss: 2.5595
Epoch 8/50 - Loss: 2.5595
Epoch 9/50 - Loss: 2.5579
Epoch 9/50 - Loss: 2.5579
Epoch 10/50 - Loss: 2.5527
Epoch 10/50 - Loss: 2.5527
Epoch 11/50 - Loss: 2.5533
Epoch 11/50 - Loss: 2.5533
Epoch 12/50 - Loss: 2.5436
Epoch 12/50 - Loss: 2.5436
Epoch 13/50 - Loss: 2.5487
Epoch 13/50 - Loss: 2.5487
Epoch 14/50 - Loss: 2.5548
Epoch 14/50 - Loss: 2.5548
Epoch 15/50 - Loss: 2.5530
Epoch 15/50 - Loss: 2.5530
Epoch 16/50 - Loss: 2.5504
Epoch 16/50 - Loss: 2.5504
Epoch 17/50 - Loss: 2.5493
Epoch 17/50 - Loss: 2.5493
Epoch 18/50 - Loss: 2.5553
Epoch 18/50 - Loss: 2.5553
Epoch 19/50 - Loss: 2.5499
Epoch 19/50 - Loss: 2.5499
Epoch 20/50 - Loss

## 9. Avaliação do Modelo

Compare a saída do modelo com as frases reais. Para avaliação semântica, pode-se usar a API do ChatGPT/Gemini (opcional).

In [57]:
# 9. Avaliação do Modelo
def evaluate_model(model, dataloader, device):
    model.eval()
    total_tokens = 0
    correct_tokens = 0
    with torch.no_grad():
        for batch in dataloader:
            inputs, targets = batch
            inputs = inputs.to(device)
            targets = targets.to(device)
            outputs = model(inputs)
            predicted = outputs.argmax(dim=-1)
            mask = (targets != 0)  # 0 é o <pad> do BERT
            correct = (predicted == targets) & mask
            correct_tokens += correct.sum().item()
            total_tokens += mask.sum().item()
    accuracy = correct_tokens / total_tokens if total_tokens > 0 else 0
    print(f"Acurácia token a token: {accuracy:.4f}")
    return accuracy

accuracy = evaluate_model(model, train_loader, device)

Acurácia token a token: 0.2637


## 10. Salvar Modelo Treinado

Salve o modelo treinado para uso futuro.

In [58]:
# 10. Salvar Modelo Treinado
torch.save(model.state_dict(), '../models/sign2text_model.pt')
print('Modelo salvo em ../models/sign2text_model.pt')

Modelo salvo em ../models/sign2text_model.pt


## 11. Integração com API GPT/Gemini (Opcional)

Utilize a API do OpenAI ou Google Gemini para polir e avaliar frases geradas.

In [ ]:
# Exemplo de integração com OpenAI GPT (opcional)
# import openai
# openai.api_key = 'SUA_API_KEY'
# def polish_phrase(phrase):
#     response = openai.ChatCompletion.create(
#         model="gpt-3.5-turbo",
#         messages=[{"role": "user", "content": f"Reescreva em português natural: {phrase}"}]
#     )
#     return response['choices'][0]['message']['content']

# frase_bruta = "voce bem"
# frase_polida = polish_phrase(frase_bruta)
# print(frase_polida)